In [1]:
import numpy as np
from pathlib import Path
import pandas as pd
from analysis_utils import get_stimulus_timestamps, get_running_timestamps, get_running_df, resample_dff_for_trial, resample_running_for_trial
from hdmf_zarr.nwb import NWBZarrIO
import matplotlib.pyplot as plt
import anndata as ad
import pickle
from scipy.interpolate import interp1d
from pathlib import Path
import os
import time
import zarr
import numcodecs
import shutil
RESAMPLE_RATE = 10    # Hz
PRE_STIM      = 1.0   # seconds before stimulus onset
POST_STIM     = 1.0   # seconds after stimulus offset

SCRATCH_DIR = Path("/root/capsule/scratch")

DATA_DIR = Path("/root/capsule/data")

load data information

In [2]:
data_info = pickle.load(open('/root/capsule/code/Preprocessing/data_info.pkl', 'rb'))
mouse_ids = list(data_info.keys())
print('mouse_ids:', mouse_ids)

mouse_ids: [778174, 786297, 797371]


Load ophys Xenium match

In [3]:
cell_types_coreg_df = pd.read_csv('/root/capsule/code/tables/cell_types_coreg.csv')
cellxgene_coreg_df = pd.read_csv('/root/capsule/code/tables/cellxgene_coreg.csv')
cellxgene_coreg_df = cellxgene_coreg_df[[col for col in cellxgene_coreg_df.columns if 'Code' not in col and 'Control' not in col]]

In [4]:
shared_columns = [col for col in cell_types_coreg_df.columns if col in cellxgene_coreg_df.columns]
cellxgene_columns = [col for col in cellxgene_coreg_df.columns if col not in shared_columns]
cell_type_columns = [col for col in cell_types_coreg_df.columns if col not in shared_columns]

# Generate Stimulus-Aligned DFF Traces (Resampled at 10 Hz)

For each mouse × session × plane × stimulus trial, extract dff traces resampled at 10 Hz in a window from **1 s before stimulus onset** to **1 s after stimulus offset**.


In [ ]:
# ── Main processing loop ────────────────────────────────────────────
# Process all mice × STAGE_1 sessions × planes × stimuli

# Output structure:
#   session_result[stim_name] = {
#       'running':       np.ndarray (trials, time, 2)  [speed, speed_filtered] float32
#       'time_relative': np.ndarray (time,)            seconds relative to stim onset
#       'trial_info':    pd.DataFrame          stimulus attributes per trial
#       'dff': {
#           plane: {'data': np.ndarray (trials, time, cells), 'cell_ids': np.ndarray (cells,)}
#       }
#   }
STIM_ALIGNED = SCRATCH_DIR / "stim_aligned_dff"
STIM_ALIGNED.mkdir(parents=True, exist_ok=True)

for mouse_id in mouse_ids:
    stage1 = {
        s: info for s, info in data_info[mouse_id]['ophys'].items()
        if info['session_type'] == 'STAGE_1'
    }

    for session_name, session_info in stage1.items():
        out_path = STIM_ALIGNED / f"{mouse_id}_{session_name}.pkl"
        if out_path.exists():
            print(f"[skip] {out_path.name} already exists")
            continue

        print(f"\n{'='*60}")
        print(f"Mouse {mouse_id}  ·  {session_name}  ·  {session_info['raw']}")
        print(f"{'='*60}")

        # ── 1. Load stimulus table & running speed ──────────────────
        ophys_raw_path = DATA_DIR / session_info['raw']

        sync_file_path = list((ophys_raw_path / 'behavior').glob("*.h5"))[0]
        stim_pkl_file_path = list((ophys_raw_path / 'behavior').glob("*.pkl"))[0]
        running_timestamps = get_running_timestamps(sync_file_path)
        running_speed_df = get_running_df(stim_pkl_file_path, running_timestamps)

        stim_table_path = list((ophys_raw_path / 'behavior').glob("*stim_table.csv"))[0]
        stimulus_table = pd.read_csv(stim_table_path)
        stim_names = stimulus_table['stim_name'].unique()
        print(f"  Stimulus table: {len(stimulus_table)} trials, types = {list(stim_names)}")

        # ── 2. Open NWB (all planes) ───────────────────────────────
        ophys_processed_path = DATA_DIR / session_info['processed']
        nwb_zarr_path = list(ophys_processed_path.glob("*.nwb"))[0]

        # Pre-build per-stim containers with stimulus-aligned running
        session_result = {}
        for stim_name in stim_names:
            stim_group = stimulus_table[stimulus_table['stim_name'] == stim_name].copy()
            stim_group = stim_group.reset_index(drop=True)
            n_trials = len(stim_group)

            # Resample running speed for each trial
            trials_running = []
            trials_time    = []
            for _, trial in stim_group.iterrows():
                run_r, t_r = resample_running_for_trial(
                    running_speed_df,
                    trial['start_time'], trial['stop_time'],
                )
                trials_running.append(run_r)
                trials_time.append(t_r)

            max_len      = max(t.shape[0] for t in trials_time)
            ref_time_idx = int(np.argmax([t.shape[0] for t in trials_time]))
            time_common  = trials_time[ref_time_idx]

            running_stacked = np.full(
                (n_trials, max_len, 2), np.nan, dtype=np.float32
            )
            for i, run_trial in enumerate(trials_running):
                running_stacked[i, :run_trial.shape[0], :] = run_trial

            session_result[stim_name] = {
                'running':       running_stacked,   # (n_trials, n_timepoints, 2) [speed, speed_filtered]
                'time_relative': time_common,
                'trial_info':    stim_group,
                'dff': {},
            }

        # ── 2b. Loop over planes and extract dff ───────────────────
        with NWBZarrIO(path=nwb_zarr_path, mode='r') as io:
            nwbfile = io.read()
        # with NWBHDF5IO(str(nwb_zarr_path), mode='r', driver='zarr') as io:
            # nwbfile = io.read()
            planes = list(nwbfile.processing.keys())

            for plane in planes:
                print(f"  Plane {plane} ... ", end="", flush=True)
                proc = nwbfile.processing[plane]['dff_timeseries']['dff_timeseries']
                dff_data   = proc.data[:]          # (n_timepoints, n_cells)
                dff_ts     = proc.timestamps[:]    # (n_timepoints,)
                cell_ids   = np.arange(1,dff_data.shape[1]+1)
                n_cells    = dff_data.shape[1]

                for stim_name in stim_names:
                    stim_group   = session_result[stim_name]['trial_info']
                    time_common  = session_result[stim_name]['time_relative']
                    n_trials     = len(stim_group)
                    max_len      = time_common.shape[0]

                    trials_dff  = []
                    trials_time = []
                    for _, trial in stim_group.iterrows():
                        dff_r, t_r = resample_dff_for_trial(
                            dff_data, dff_ts,
                            trial['start_time'], trial['stop_time'],
                        )
                        trials_dff.append(dff_r)
                        trials_time.append(t_r)

                    # Pad to longest time grid
                    dff_stacked = np.full(
                        (n_trials, max_len, n_cells), np.nan, dtype=np.float32
                    )
                    for i, dff_trial in enumerate(trials_dff):
                        dff_stacked[i, :dff_trial.shape[0], :] = dff_trial

                    # Store per-plane dff
                    session_result[stim_name]['dff'][plane] = {
                        'data':     dff_stacked,
                        'cell_ids': cell_ids,
                    }

                shapes_str = ", ".join(
                    f"{sn}: {session_result[sn]['dff'][plane]['data'].shape}"
                    for sn in stim_names
                )
                print(f"done  [{shapes_str}]")

        # ── 3. Save per-session pickle ──────────────────────────────
        with open(out_path, 'wb') as f:
            pickle.dump(session_result, f, protocol=pickle.HIGHEST_PROTOCOL)
        size_mb = out_path.stat().st_size / 1e6

print("\n✅ All sessions processed.")


Mouse 778174  ·  session_1  ·  multiplane-ophys_778174_2025-03-14_11-17-01


/opt/conda/envs/ophys_env/lib/python3.12/site-packages/pandas/compat/pickle_compat.py:35: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  stack[-1] = func(*args)


  Stimulus table: 1124 trials, types = ['spontaneous', 'drifting_gratings_contrast', 'drifting_gratings_TF']
  Plane VISp_0 ... done  [spontaneous: (4, 3634, 162), drifting_gratings_contrast: (560, 41, 162), drifting_gratings_TF: (560, 41, 162)]
  Plane VISp_1 ... done  [spontaneous: (4, 3634, 253), drifting_gratings_contrast: (560, 41, 253), drifting_gratings_TF: (560, 41, 253)]
  Plane VISp_2 ... done  [spontaneous: (4, 3634, 432), drifting_gratings_contrast: (560, 41, 432), drifting_gratings_TF: (560, 41, 432)]
  Plane VISp_3 ... done  [spontaneous: (4, 3634, 300), drifting_gratings_contrast: (560, 41, 300), drifting_gratings_TF: (560, 41, 300)]
  Plane VISp_4 ... done  [spontaneous: (4, 3634, 428), drifting_gratings_contrast: (560, 41, 428), drifting_gratings_TF: (560, 41, 428)]
  Plane VISp_5 ... done  [spontaneous: (4, 3634, 344), drifting_gratings_contrast: (560, 41, 344), drifting_gratings_TF: (560, 41, 344)]
  Plane VISp_6 ... done  [spontaneous: (4, 3634, 443), drifting_grati

/opt/conda/envs/ophys_env/lib/python3.12/site-packages/pandas/compat/pickle_compat.py:35: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  stack[-1] = func(*args)


  Stimulus table: 1124 trials, types = ['spontaneous', 'drifting_gratings_contrast', 'drifting_gratings_TF']
  Plane VISp_0 ... done  [spontaneous: (4, 3634, 149), drifting_gratings_contrast: (560, 41, 149), drifting_gratings_TF: (560, 41, 149)]
  Plane VISp_1 ... done  [spontaneous: (4, 3634, 219), drifting_gratings_contrast: (560, 41, 219), drifting_gratings_TF: (560, 41, 219)]
  Plane VISp_2 ... done  [spontaneous: (4, 3634, 399), drifting_gratings_contrast: (560, 41, 399), drifting_gratings_TF: (560, 41, 399)]
  Plane VISp_3 ... done  [spontaneous: (4, 3634, 264), drifting_gratings_contrast: (560, 41, 264), drifting_gratings_TF: (560, 41, 264)]
  Plane VISp_4 ... done  [spontaneous: (4, 3634, 398), drifting_gratings_contrast: (560, 41, 398), drifting_gratings_TF: (560, 41, 398)]
  Plane VISp_5 ... done  [spontaneous: (4, 3634, 340), drifting_gratings_contrast: (560, 41, 340), drifting_gratings_TF: (560, 41, 340)]
  Plane VISp_6 ... done  [spontaneous: (4, 3634, 407), drifting_grati

...

  Stimulus table: 1124 trials, types = ['spontaneous', 'drifting_gratings_contrast', 'drifting_gratings_TF']
  Plane VISp_0 ... done  [spontaneous: (4, 3634, 439), drifting_gratings_contrast: (560, 41, 439), drifting_gratings_TF: (560, 41, 439)]
  Plane VISp_1 ... done  [spontaneous: (4, 3634, 431), drifting_gratings_contrast: (560, 41, 431), drifting_gratings_TF: (560, 41, 431)]
  Plane VISp_2 ... done  [spontaneous: (4, 3634, 427), drifting_gratings_contrast: (560, 41, 427), drifting_gratings_TF: (560, 41, 427)]
  Plane VISp_3 ... done  [spontaneous: (4, 3634, 501), drifting_gratings_contrast: (560, 41, 501), drifting_gratings_TF: (560, 41, 501)]
  Plane VISp_4 ... done  [spontaneous: (4, 3634, 299), drifting_gratings_contrast: (560, 41, 299), drifting_gratings_TF: (560, 41, 299)]
  Plane VISp_5 ... done  [spontaneous: (4, 3634, 491), drifting_gratings_contrast: (560, 41, 491), drifting_gratings_TF: (560, 41, 491)]
  Plane VISp_6 ... done  [spontaneous: (4, 3634, 71), drifting_gratin

/opt/conda/envs/ophys_env/lib/python3.12/site-packages/pandas/compat/pickle_compat.py:35: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  stack[-1] = func(*args)


  Stimulus table: 1124 trials, types = ['spontaneous', 'drifting_gratings_contrast', 'drifting_gratings_TF']
  Plane VISp_0 ... done  [spontaneous: (4, 3634, 454), drifting_gratings_contrast: (560, 41, 454), drifting_gratings_TF: (560, 41, 454)]
  Plane VISp_1 ... done  [spontaneous: (4, 3634, 463), drifting_gratings_contrast: (560, 41, 463), drifting_gratings_TF: (560, 41, 463)]
  Plane VISp_2 ... done  [spontaneous: (4, 3634, 451), drifting_gratings_contrast: (560, 41, 451), drifting_gratings_TF: (560, 41, 451)]
  Plane VISp_3 ... done  [spontaneous: (4, 3634, 510), drifting_gratings_contrast: (560, 41, 510), drifting_gratings_TF: (560, 41, 510)]
  Plane VISp_4 ... done  [spontaneous: (4, 3634, 298), drifting_gratings_contrast: (560, 41, 298), drifting_gratings_TF: (560, 41, 298)]
  Plane VISp_5 ... done  [spontaneous: (4, 3634, 494), drifting_gratings_contrast: (560, 41, 494), drifting_gratings_TF: (560, 41, 494)]
  Plane VISp_6 ... done  [spontaneous: (4, 3634, 58), drifting_gratin

/opt/conda/envs/ophys_env/lib/python3.12/site-packages/pandas/compat/pickle_compat.py:35: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  stack[-1] = func(*args)


  Stimulus table: 1124 trials, types = ['spontaneous', 'drifting_gratings_contrast', 'drifting_gratings_TF']
  Plane VISp_0 ... done  [spontaneous: (4, 3634, 456), drifting_gratings_contrast: (560, 41, 456), drifting_gratings_TF: (560, 41, 456)]
  Plane VISp_1 ... done  [spontaneous: (4, 3634, 448), drifting_gratings_contrast: (560, 41, 448), drifting_gratings_TF: (560, 41, 448)]
  Plane VISp_2 ... done  [spontaneous: (4, 3634, 457), drifting_gratings_contrast: (560, 41, 457), drifting_gratings_TF: (560, 41, 457)]
  Plane VISp_3 ... done  [spontaneous: (4, 3634, 509), drifting_gratings_contrast: (560, 41, 509), drifting_gratings_TF: (560, 41, 509)]
  Plane VISp_4 ... done  [spontaneous: (4, 3634, 304), drifting_gratings_contrast: (560, 41, 304), drifting_gratings_TF: (560, 41, 304)]
  Plane VISp_5 ... done  [spontaneous: (4, 3634, 474), drifting_gratings_contrast: (560, 41, 474), drifting_gratings_TF: (560, 41, 474)]
  Plane VISp_6 ... done  [spontaneous: (4, 3634, 66), drifting_gratin

In [12]:
# ── Coregister: subset to matched cells ─────────────────────────────
for mouse_id in mouse_ids:
    all_sessions_data = {}
    out_path = RESULTS_DIR / f"{mouse_id}_multimodal_data.pkl"
    mouse_cell_ids = cell_types_coreg_df.loc[cell_types_coreg_df['mouse_id']==mouse_id, 'unique_id']
    cell_types_table = cell_types_coreg_df.set_index('unique_id').loc[mouse_cell_ids,cell_type_columns]
    cellxgene_table = cellxgene_coreg_df.set_index('unique_id').loc[mouse_cell_ids,cellxgene_columns]
    all_sessions_data['unique_id'] = mouse_cell_ids
    all_sessions_data['transcriptomics'] = {'cellxgene':cellxgene_table, 'cell_type':cell_types_table}
    all_sessions_data['ophys'] = {'drifting_gratings':{}}
    for session_name in ['session_1', 'session_2', 'session_3']:
        in_path  = RESULTS_DIR / f"{mouse_id}_{session_name}.pkl"
        
        with open(in_path, 'rb') as f:
            session_data = pickle.load(f)

        for stim_name in session_data:
            planes = list(session_data[stim_name]['dff'].keys())
            all_dff = []
            all_unique_ids = []
            for plane in planes:
                cell_ids_all = session_data[stim_name]['dff'][plane]['cell_ids']

                coreg_mask = (
                    (cell_types_coreg_df['mouse_id'] == mouse_id) &
                    (cell_types_coreg_df['plane'] == plane)
                )
                cell_ids_coreg = cell_types_coreg_df.loc[coreg_mask, f'cell_id - {session_name}'].values
                unique_ids     = cell_types_coreg_df.loc[coreg_mask, 'unique_id'].values

                cell_id_locs = np.array([
                    np.argwhere(cell_ids_all == cid)[0, 0]
                    for cid in cell_ids_coreg
                ]).astype(int)
                all_dff.append(session_data[stim_name]['dff'][plane]['data'][..., cell_id_locs])
                all_unique_ids.append(unique_ids)
            all_unique_ids = np.concatenate(all_unique_ids, axis=-1)
            session_data[stim_name]['dff'] = np.concatenate(all_dff, axis=-1)[...,np.argwhere(np.isin(mouse_cell_ids,all_unique_ids))[:,0]]
        
        all_sessions_data['ophys']["drifting_gratings"][session_name] = session_data
    with open(out_path, 'wb') as f:
        pickle.dump(all_sessions_data, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"✓ {out_path.name}")

print("\n✅ All coregistered files saved.")

✓ 778174_multimodal_data.pkl
✓ 786297_multimodal_data.pkl
✓ 797371_multimodal_data.pkl

✅ All coregistered files saved.


In [5]:
glm_keys = ['alpha', 'coef', 'score', 'y', 'y_hat']
STIM_ALIGNED = SCRATCH_DIR / "stim_aligned_dff"
MORPH_DIR = SCRATCH_DIR / "morphology"
MULTIMODAL = SCRATCH_DIR / "multimodal_data"
MULTIMODAL.mkdir(parents=True, exist_ok=True)

def _safe_create_dataset(group, name, max_retries=5, backoff=0.5, **kwargs):
    """Retry group.create_dataset on transient FileNotFoundError (zarr partial-rename races).
    Also handle zarr.errors.ContainsArrayError by removing the conflicting array and retrying."""
    for attempt in range(1, max_retries + 1):
        try:
            return group.create_dataset(name, **kwargs)
        except FileNotFoundError:
            if attempt == max_retries:
                raise
            time.sleep(backoff * attempt)
        except zarr.errors.ContainsArrayError:
            # If a zarr Array exists where a Group is expected for the path component,
            # remove the conflicting array so we can recreate the intended dataset.
            try:
                del group[name]
            except Exception:
                # best-effort removal; we'll retry until max_retries
                pass
            if attempt == max_retries:
                raise
            time.sleep(backoff * attempt)

for mi, mouse_id in enumerate(mouse_ids):
    out_path = MULTIMODAL / f"{mouse_id}_multimodal_data.zarr"
    print(f"\n{'='*60}")
    print(f"[{mi+1}/{len(mouse_ids)}] Processing mouse {mouse_id}")
    print(f"{'='*60}")
    
    mouse_cell_ids: np.ndarray = cell_types_coreg_df.loc[
        cell_types_coreg_df['mouse_id'] == mouse_id, 'unique_id'
    ].values

    cell_types_table: pd.DataFrame = cell_types_coreg_df.set_index('unique_id').loc[mouse_cell_ids, cell_type_columns]
    cellxgene_table: pd.DataFrame = cellxgene_coreg_df.set_index('unique_id').loc[mouse_cell_ids, cellxgene_columns]

    # ── Open or create Zarr store ───────────────────────────────────
    store = zarr.DirectoryStore(str(out_path))
    if out_path.exists():
        print("  → Existing Zarr store found – resuming...")
        root = zarr.open_group(store, mode='a')
    else:
        print("  → Creating new Zarr store...")
        root = zarr.group(store, overwrite=True)

    # ── Save unique_id ──────────────────────────────────────────────
    if 'unique_id' in root:
        print("  → unique_id already exists – skipping.")
    else:
        print("  → Saving unique_id array...")
        _ = root.array('unique_id', data=mouse_cell_ids, dtype=str,
                   object_codec=numcodecs.VLenUTF8())

    # ── Save transcriptomics tables ─────────────────────────────────
    if 'transcriptomics' not in root:
        root.create_group('transcriptomics')
    transcriptomics_group: zarr.Group = root['transcriptomics']  # type: ignore[assignment]

    # cell_type
    if 'cell_type' in transcriptomics_group:
        print("  → transcriptomics/cell_type already exists – skipping.")
    else:
        print("  → Saving transcriptomics / cell_type...")
        ct_group = transcriptomics_group.create_group('cell_type')
        ct_group.attrs['index'] = list(cell_types_table.index)
        ct_group.attrs['columns'] = list(cell_types_table.columns)
        for col in cell_types_table.columns:
            vals = cell_types_table[col].values
            if vals.dtype == object:
                _ = ct_group.array(str(col), data=vals, dtype=str,
                               object_codec=numcodecs.VLenUTF8())
            else:
                _ = ct_group.array(str(col), data=vals)

    # cellxgene
    if 'cellxgene' in transcriptomics_group:
        print("  → transcriptomics/cellxgene already exists – skipping.")
    else:
        print("  → Saving transcriptomics / cellxgene...")
        cxg_group = transcriptomics_group.create_group('cellxgene')
        cxg_group.attrs['index'] = list(cellxgene_table.index)
        cxg_group.attrs['columns'] = list(cellxgene_table.columns)
        for col in cellxgene_table.columns:
            vals = cellxgene_table[col].values
            if vals.dtype == object:
                _ = cxg_group.array(str(col), data=vals, dtype=str,
                                object_codec=numcodecs.VLenUTF8())
            else:
                _ = cxg_group.array(str(col), data=vals)

    # ── Save morphology tables ─────────────────────────────────
    if 'morphology' not in root:
        root.create_group('morphology')
    morph_group = root['morphology']  # type: ignore[assignment]

    if 'mask_properties' in morph_group:
        print("  → morphology/mask_properties already exists – skipping.")
    else:
        print("  → Adding morphology / mask_properties...")
        mask_prop_group = morph_group.create_group('mask_properties')

        morph_csv = MORPH_DIR / f"{mouse_id}_mask_properties.csv"
        print(f"        Loading morphology file: {morph_csv.name}...")
        try:
            df_masks_props = pd.read_csv(morph_csv)
        except FileNotFoundError:
            print(f"        → File not found: {morph_csv} — skipping morphology for mouse {mouse_id}.")
        else:
            print("        Subsetting mask properties to coregistered cells...")
        coreg_mask = (cell_types_coreg_df['mouse_id'] == mouse_id)
        mask_labels = cell_types_coreg_df.loc[coreg_mask, 'cell_id - zstack'].values
        unique_id = cell_types_coreg_df.loc[coreg_mask, 'unique_id'].values
        df_masks_props = df_masks_props.set_index('label').loc[mask_labels]
        df_masks_props['unique_id'] = unique_id
        df_masks_props.set_index('unique_id', inplace=True, drop=True)

        print("        Writing mask properties to Zarr...")
        mask_prop_group.attrs['index'] = list(df_masks_props.index)
        mask_prop_group.attrs['columns'] = list(df_masks_props.columns)
        mask_prop_group.attrs['n_rows'] = len(df_masks_props)
        for col in df_masks_props.columns:
            vals = df_masks_props[col].values
            if vals.dtype == object:
                _ = mask_prop_group.array(str(col), data=vals, dtype=str,
                            object_codec=numcodecs.VLenUTF8())
            else:
                _ = mask_prop_group.array(str(col), data=vals)


    # ── Save ophys data per session × stimulus ──────────────────────
    if 'ophys' not in root:
        root.create_group('ophys')
    ophys_group: zarr.Group = root['ophys']  # type: ignore[assignment]

    if 'drifting_gratings' not in ophys_group:
        ophys_group.create_group('drifting_gratings')
    dg_group: zarr.Group = ophys_group['drifting_gratings']  # type: ignore[assignment]

    for si, session_name in enumerate(['session_1', 'session_2', 'session_3']):
        print(f"  → [{si+1}/3] Processing {session_name}...")

        if session_name not in dg_group:
            dg_group.create_group(session_name)
        sess_group: zarr.Group = dg_group[session_name]  # type: ignore[assignment]

        # ── stim_aligned_dff ────────────────────────────────────────
        if 'stim_aligned_dff' not in sess_group:
            sess_group.create_group('stim_aligned_dff')
        sadff_group: zarr.Group = sess_group['stim_aligned_dff']  # type: ignore[assignment]

        in_path = STIM_ALIGNED / f"{mouse_id}_{session_name}.pkl"

        # Load session data
        print(f"      → Loading {in_path.name}...")
        with open(in_path, 'rb') as f:
            session_data = pickle.load(f)

        stim_names: list[str] = list(session_data.keys())

        for sti, stim_name in enumerate(stim_names):
            if stim_name in sadff_group:
                # Quick check: does the group already have 'dff'?
                stim_grp_existing: zarr.Group = sadff_group[stim_name]  # type: ignore[assignment]
                if 'dff' in stim_grp_existing and 'trial_info' in stim_grp_existing:
                    print(f"      → [{sti+1}/{len(stim_names)}] Stimulus {stim_name} already exists – skipping.")
                    continue

            print(f"      → [{sti+1}/{len(stim_names)}] Stimulus: {stim_name}")
            sd: dict = session_data
            planes: list[str] = list(sd[stim_name]['dff'].keys())
            all_dff: list[np.ndarray] = []
            all_unique_ids_list: list[np.ndarray] = []

            for plane in planes:
                cell_ids_all = sd[stim_name]['dff'][plane]['cell_ids']
                coreg_mask = (
                    (cell_types_coreg_df['mouse_id'] == mouse_id) &
                    (cell_types_coreg_df['plane'] == plane)
                )
                cell_ids_coreg: np.ndarray = cell_types_coreg_df.loc[
                    coreg_mask, f'cell_id - {session_name}'
                ].values
                unique_ids_plane: np.ndarray = cell_types_coreg_df.loc[
                    coreg_mask, 'unique_id'
                ].values

                cell_id_locs = np.array([
                    np.argwhere(cell_ids_all == cid)[0, 0]
                    for cid in cell_ids_coreg
                ]).astype(int)

                all_dff.append(
                    sd[stim_name]['dff'][plane]['data'][..., cell_id_locs]
                )
                all_unique_ids_list.append(unique_ids_plane)

            all_unique_ids: np.ndarray = np.concatenate(all_unique_ids_list, axis=-1)
            dff_coreg: np.ndarray = np.concatenate(all_dff, axis=-1)[
                ..., np.argwhere(np.isin(mouse_cell_ids, all_unique_ids))[:, 0]
            ]

            # Write stimulus group (remove partial if exists)
            if stim_name in sadff_group:
                del sadff_group[stim_name]
            stim_group = sadff_group.create_group(stim_name)

            # print(f"        Writing dff {dff_coreg.shape}...")
            # _ = stim_group.create_dataset(
            #     'dff', data=dff_coreg, dtype='float32',
            #     chunks=(1, dff_coreg.shape[1], min(int(dff_coreg.shape[2]), 64)),
            #     compressor=numcodecs.Blosc(cname='zstd', clevel=3),
            # )

            print(f"        Writing dff {dff_coreg.shape}...")
            _ = _safe_create_dataset(
                stim_group, 'dff', data=dff_coreg, dtype='float32',
                chunks=(1, dff_coreg.shape[1], min(int(dff_coreg.shape[2]), 64)),
                compressor=numcodecs.Blosc(cname='zstd', clevel=3),
            )

            print(f"        Writing running...")
            running_data = sd[stim_name]['running']
            _ = stim_group.array(
                'running',
                data=running_data,
                dtype='float32',
                chunks=(1, running_data.shape[1], 2),
                compressor=numcodecs.Blosc(cname='zstd', clevel=3),
            )

            _ = stim_group.array(
                'time_relative',
                data=sd[stim_name]['time_relative'],
            )

            print(f"        Writing trial_info...")
            trial_info: pd.DataFrame = sd[stim_name]['trial_info']
            ti_group = stim_group.create_group('trial_info')
            ti_group.attrs['columns'] = list(trial_info.columns)
            ti_group.attrs['n_rows'] = len(trial_info)
            for col in trial_info.columns:
                vals = trial_info[col].values
                if vals.dtype == object:
                    _ = ti_group.array(str(col), data=vals, dtype=str,
                                   object_codec=numcodecs.VLenUTF8())
                else:
                    _ = ti_group.array(str(col), data=vals)

        # ── GLM data ───────────────────────────────────────────────
        if 'glm' not in sess_group:
            sess_group.create_group('glm')
        glm_group: zarr.Group = sess_group['glm']  # type: ignore[assignment]

        glm_path = DATA_DIR / data_info[mouse_id]['ophys'][session_name]['glm']
        glm_files: list[Path] = list(glm_path.glob('*.hdf'))

        # Build df for cell_id mapping (needed for GLM)
        df: pd.DataFrame = cell_types_coreg_df.set_index('unique_id')
        # Compute unique_ids for this mouse
        mouse_unique_ids: np.ndarray = cell_types_coreg_df.loc[
            cell_types_coreg_df['mouse_id'] == mouse_id, 'unique_id'
        ].values

        for ki, key in enumerate(glm_keys):
            if key in glm_group:
                # Check it has at least one array inside
                key_grp_existing: zarr.Group = glm_group[key]  # type: ignore[assignment]
                if len(list(key_grp_existing.arrays())) > 0:
                    print(f"        GLM key '{key}' already exists – skipping.")
                    continue

            print(f"        [{ki+1}/{len(glm_keys)}] GLM key: {key}")
            glm_output_parts: list[pd.DataFrame] = []
            for glm_file in glm_files:
                glm_output_parts.append(pd.read_hdf(str(glm_file), key=key))
            glm_output_total: pd.DataFrame = pd.concat(glm_output_parts, axis=0)
            glm_output_total = glm_output_total.set_index(glm_output_total.index)

            cell_ids: list[str] = [
                f"{row['plane']}_{int(row[f'cell_id - {session_name}'])-1}"
                for _, row in df.loc[mouse_unique_ids].iterrows()
            ]
            glm_output_total = glm_output_total.loc[cell_ids].reset_index(drop=True)
            print(glm_output_total.shape, len(mouse_unique_ids))
            # Remove partial if exists
            if key in glm_group:
                del glm_group[key]
            key_group = glm_group.create_group(key)

            key_group.attrs['columns'] = list(glm_output_total.columns)
            key_group.attrs['n_rows'] = len(glm_output_total)
            for col in glm_output_total.columns:
                vals = glm_output_total[col].values
                if vals.dtype == object:
                    first_valid = next((v for v in vals if v is not None), None)
                    if first_valid is not None and isinstance(first_valid, (list, np.ndarray)):
                        stacked: np.ndarray = np.array(
                            [np.asarray(v, dtype=np.float32) for v in vals]
                        )
                        _ = key_group.array(str(col), data=stacked, dtype='float32')
                    else:
                        str_vals: np.ndarray = np.array(
                            [str(v) for v in vals], dtype=object
                        )
                        _ = key_group.array(str(col), data=str_vals, dtype=str,
                                      object_codec=numcodecs.VLenUTF8())
                else:
                    _ = key_group.array(str(col), data=vals)
        




    zarr.consolidate_metadata(store)
    print(f"  ✓ {out_path.name} saved.")

print("\n✅ All coregistered files saved as Zarr.")


[1/3] Processing mouse 778174
  → Existing Zarr store found – resuming...
  → unique_id already exists – skipping.
  → transcriptomics/cell_type already exists – skipping.
  → transcriptomics/cellxgene already exists – skipping.
  → morphology/mask_properties already exists – skipping.
  → [1/3] Processing session_1...
      → Loading 778174_session_1.pkl...
      → [1/3] Stimulus GreyScreen already exists – skipping.
      → [2/3] Stimulus GratingStim already exists – skipping.
      → [3/3] Stimulus Catch already exists – skipping.
        [1/5] GLM key: alpha
(614, 1) 614
        [2/5] GLM key: coef
(614, 167) 614
        [3/5] GLM key: score
(614, 171) 614
        [4/5] GLM key: y
(614, 1) 614
        [5/5] GLM key: y_hat
(614, 3) 614
  → [2/3] Processing session_2...
      → Loading 778174_session_2.pkl...
      → [1/3] Stimulus GreyScreen already exists – skipping.
      → [2/3] Stimulus GratingStim already exists – skipping.
      → [3/3] Stimulus Catch already exists – skippin

- adata.X ← avergae df/f during 2s of stimuli
- adata.layers["baseline"] ← average df/f during 1s gray screenbefore the sitmuli
- adata.obs ← cell metadata
- adata.var ← trial metadata

In [21]:
stim_type = 'GratingStim'
running_thr = 1.0
for mouse_id in mouse_ids:
    X = []
    UID = []
    VAR = []
    for session_name in ['session_1', 'session_2', 'session_3']:
        in_path = STIM_ALIGNED / f"{mouse_id}_{session_name}_coregistered.pkl"
        with open(in_path, 'rb') as f:
            session_data = pickle.load(f)

        var = session_data[stim_type]['trial_info']
        var = var[~var['gray_screen']].reset_index(drop=True)
        target_slice = session_data[stim_type]['running'][1::2,10:-10,0]
        if np.count_nonzero(~np.isnan(target_slice)) == 0:
            avg_running = np.nan
        else:
            avg_running = np.nanmean(target_slice, axis=1)
        print(session_data[stim_type]['running'].shape)
        var['avg_running'] = avg_running
        var['is_running'] = avg_running>running_thr
        var['day'] = int(session_name.split('_')[-1])
        var['date'] = data_info[mouse_id]['ophys'][session_name]['raw'].split('_')[-2]
        VAR.append(var)
        x1 = np.nanmean(session_data[stim_type]['dff']['data'][1::2,10:-10], axis=1)
        x2 = np.nanmean(session_data[stim_type]['dff']['data'][::2,10:-10], axis=1)
        X.append(np.stack([x1, x2]))
        UID.append(session_data[stim_type]['dff']['unique_ids'])

    VAR = pd.concat(VAR, axis=0).reset_index(drop=True)
    X = np.concatenate(X, axis=1)
    assert all(np.array_equal(UID[0], u) for u in UID[1:]), "UID vectors are not equal across sessions"
    print("✓ All UID vectors are equal")
    UID = UID[0]
    OBS = cell_types_coreg_df.merge(cellxgene_coreg_df, on=[col for col in cell_types_coreg_df.columns if col in cellxgene_coreg_df.columns]).set_index('unique_id',drop=True).loc[UID].reset_index()
    adata = ad.AnnData(
        X=X[0].T,
        obs=OBS.reset_index(drop=True),
        var=VAR.reset_index(drop=True),
        layers={"baseline": X[1].T},
    )

    out_path = STIM_ALIGNED / f"{mouse_id}_GratingStim.h5ad"
    adata.write_h5ad(out_path)
    print(adata)
    print(f"Saved to {out_path}")

FileNotFoundError: [Errno 2] No such file or directory: '/root/capsule/scratch/stim_aligned_dff/778174_session_1_coregistered.pkl'

In [ ]:
CHECKPOINT_INTERVAL = 5 * 60  # seconds between timed checkpoints

for mouse_id in mouse_ids:
    in_path = STIM_ALIGNED / f"{mouse_id}_multimodal_data.pkl"
    with open(in_path, 'rb') as f:
            session_data = pickle.load(f)
    for session_name in ['session_1', 'session_2', 'session_3']:
        
        out_path = STIM_ALIGNED / f"{mouse_id}_{session_name}_correlation_matrix.pkl"

        # ── Load existing partial results if available ──────────────
        if out_path.exists():
            with open(out_path, 'rb') as f:
                corr_results = pickle.load(f)
            print(f"[resume] Loaded partial results from {out_path.name}")
        else:
            corr_results = {}

        

        dirty = False  # track whether we need to save
        last_save_time = time.time()

        def _timed_checkpoint(force=False):
            """Save corr_results if ≥ CHECKPOINT_INTERVAL seconds elapsed (or force=True)."""
            global last_save_time, dirty
            now = time.time()
            if force or (now - last_save_time >= CHECKPOINT_INTERVAL):
                with open(out_path, 'wb') as f:
                    pickle.dump(corr_results, f, protocol=pickle.HIGHEST_PROTOCOL)
                elapsed = int(now - last_save_time)
                print(f"  [checkpoint @ {elapsed}s] saved {out_path.name}")
                last_save_time = now
                dirty = False

        for stim_name in session_data['ophys']['drifting_gratings'][session_name]:
            if stim_name not in corr_results:
                corr_results[stim_name] = {}
            dff = session_data['ophys']['drifting_gratings'][session_name][stim_name]['dff']
            n_cells: int = dff.shape[2]

            if stim_name == "GratingStim":
                # continue
                trial_info = session_data['ophys']['drifting_gratings'][session_name][stim_name]['trial_info']
                combinations = trial_info.drop_duplicates(['stim_name', 'contrast','temporal_frequency','orientation']).\
                    sort_values(['stim_name', 'contrast','temporal_frequency','orientation'])[['stim_name', 'contrast','temporal_frequency','orientation']].reset_index(drop = True)
                for combination in combinations.itertuples(index=False):
                    combo_key = f"{combination.stim_name.replace('drifting_gratings','sweeping')}_{int(combination.contrast*100)}%_{int(combination.temporal_frequency)}Hz_{int(combination.orientation)}°"

                    # Check for fully computed or partial result
                    start_row = 0
                    if combo_key in corr_results[stim_name]:
                        existing = corr_results[stim_name][combo_key]
                        if isinstance(existing, dict):
                            # Partial result: {'matrix': ..., 'completed_rows': N}
                            start_row = existing['completed_rows']
                            combo_corr = existing['matrix']
                            print(f"[resume] {combo_key} from row {start_row}/{n_cells}")
                        elif isinstance(existing, np.ndarray) and existing.shape == (n_cells, n_cells):
                            print(f"[skip] {combo_key} already computed")
                            continue
                    else:
                        combo_corr = np.zeros((n_cells, n_cells))

                    stim_inds = trial_info[
                        (trial_info['stim_name'] == combination.stim_name) &
                        (trial_info['contrast'] == combination.contrast) &
                        (trial_info['temporal_frequency'] == combination.temporal_frequency) &
                        (trial_info['orientation'] == combination.orientation)
                    ].index.values[::2].astype(int)

                    print(f"Computing {combo_key} ({n_cells} cells, from row {start_row}) ...", end=" ", flush=True)

                    for i in range(start_row, n_cells):
                        for j in range(n_cells):
                            combo_corr[i, j] = np.nanmean([np.corrcoef(dff[int(si), 10:-10, i], dff[int(si), 10:-10, j])[0, 1] for si in stim_inds])
                            if np.isnan(combo_corr[i, j]):
                                print(combo_key, i, j, 'nan')

                        # Timed partial checkpoint (save progress per-row)
                        if time.time() - last_save_time >= CHECKPOINT_INTERVAL:
                            corr_results[stim_name][combo_key] = {'matrix': combo_corr, 'completed_rows': i + 1}
                            dirty = True
                            _timed_checkpoint(force=True)

                    # Store final complete matrix (plain ndarray, not dict)
                    corr_results[stim_name][combo_key] = combo_corr
                    dirty = True
                    print("done")
                    _timed_checkpoint(force=True)

            # ── "all" correlation matrix ────────────────────────────
            start_row = 0
            if 'all' in corr_results[stim_name]:
                existing = corr_results[stim_name]['all']
                if isinstance(existing, dict):
                    start_row = existing['completed_rows']
                    correlation_matrix = existing['matrix']
                    print(f"[resume] {stim_name}/all from row {start_row}/{n_cells}")
                elif isinstance(existing, np.ndarray) and existing.shape == (n_cells, n_cells):
                    print(f"[skip] {stim_name}/all already computed")
                    continue
            else:
                correlation_matrix = np.zeros((n_cells, n_cells))

            print(f"Computing {stim_name}/all ({n_cells} cells, from row {start_row}) ...", end=" ", flush=True)
            n_trials = dff.shape[0]
            for i in range(start_row, n_cells):
                for j in range(n_cells):
                    correlation_matrix[i, j] = np.nanmean([np.corrcoef(dff[int(stim_ind), 10:-10, i], dff[int(stim_ind), 10:-10, j])[0, 1] for stim_ind in range(n_trials)])

                # Timed partial checkpoint
                if time.time() - last_save_time >= CHECKPOINT_INTERVAL:
                    corr_results[stim_name]['all'] = {'matrix': correlation_matrix, 'completed_rows': i + 1}
                    dirty = True
                    _timed_checkpoint(force=True)

            corr_results[stim_name]['all'] = correlation_matrix
            dirty = True
            print("done")
            _timed_checkpoint(force=True)

        if dirty:
            _timed_checkpoint(force=True)
            print(f"✓ {out_path.name} (complete)")
        else:
            print(f"✓ {out_path.name} (nothing new to compute)")

[resume] Loaded partial results from 778174_session_1_correlation_matrix.pkl
[skip] GreyScreen/all already computed
[skip] sweeping_TF_80%_1Hz_0° already computed
[skip] sweeping_TF_80%_1Hz_45° already computed
[skip] sweeping_TF_80%_1Hz_90° already computed
[skip] sweeping_TF_80%_1Hz_135° already computed
[skip] sweeping_TF_80%_1Hz_180° already computed
[skip] sweeping_TF_80%_1Hz_225° already computed
[skip] sweeping_TF_80%_1Hz_270° already computed
[skip] sweeping_TF_80%_1Hz_315° already computed
[skip] sweeping_TF_80%_2Hz_0° already computed
[skip] sweeping_TF_80%_2Hz_45° already computed
[skip] sweeping_TF_80%_2Hz_90° already computed
[skip] sweeping_TF_80%_2Hz_135° already computed
[skip] sweeping_TF_80%_2Hz_180° already computed
[skip] sweeping_TF_80%_2Hz_225° already computed
[skip] sweeping_TF_80%_2Hz_270° already computed
[skip] sweeping_TF_80%_2Hz_315° already computed
[skip] sweeping_TF_80%_4Hz_0° already computed
[skip] sweeping_TF_80%_4Hz_45° already computed
[skip] sweepi

In [15]:
session_data.keys()

dict_keys(['unique_id', 'transcriptomics', 'ophys'])

In [17]:
session_data['transcriptomics'].keys()

dict_keys(['cellxgene', 'cell_type'])

In [18]:
session_data['ophys'].keys()

dict_keys(['drifting_gratings'])

In [19]:
session_data['ophys']['drifting_gratings'].keys()

dict_keys(['session_1', 'session_2', 'session_3'])

In [21]:
session_data['ophys']['drifting_gratings']['session_3'].keys()

dict_keys(['GreyScreen', 'GratingStim', 'Catch'])

In [23]:
session_data['ophys']['drifting_gratings']['session_3']['GratingStim'].keys()

dict_keys(['running', 'time_relative', 'trial_info', 'dff'])

In [24]:
session_data['ophys']['drifting_gratings']['session_3']['GratingStim']['dff'].shape

(2186, 41, 614)

In [25]:
session_data['ophys']['drifting_gratings']['session_3']['GratingStim']['trial_info'].shape

(2186, 13)

In [26]:
session_data['ophys']['drifting_gratings']['session_3']['GratingStim']['time_relative'].shape

(41,)

In [27]:
session_data['ophys']['drifting_gratings']['session_3']['GratingStim']['running'].shape

(2186, 41, 2)